In [ ]:
import pandas as pd
import numpy as np

# Load dataset
df = pd.read_csv('/telco.csv')

# ---------------------------------------------------------
# 1. STANDARDIZE COLUMN NAMES
# Why: Spaces in column names cause errors later in SQL exports
# and make Python code harder to write (df['Customer ID'] vs df.CustomerID)
# ---------------------------------------------------------
df.columns = df.columns.str.strip().str.replace(' ', '_')

# ---------------------------------------------------------
# 2. HANDLE STRUCTURAL MISSING VALUES
# Why: These nulls represent "not applicable," not missing data.
# Leaving them as NaN would break groupby/filter logic in EDA and SQL.
# ---------------------------------------------------------
df['Internet_Type'] = df['Internet_Type'].fillna('No Internet')
df['Offer'] = df['Offer'].fillna('No Offer')

# Churn Category/Reason: only fill for reporting clarity, but we keep
# a boolean flag so we never accidentally treat "No Churn" as a real category
df['Churn_Category'] = df['Churn_Category'].fillna('Not Churned')
df['Churn_Reason'] = df['Churn_Reason'].fillna('Not Churned')

# ---------------------------------------------------------
# 3. REMOVE DUPLICATES (defensive step)
# Why: Even though we confirmed 0 duplicates, always include this
# in production-style pipelines in case source data changes.
# ---------------------------------------------------------
df = df.drop_duplicates(subset='Customer_ID')

# ---------------------------------------------------------
# 4. FIX DATA TYPES
# Why: Correct dtypes reduce memory, prevent silent bugs in
# aggregation, and let Power BI/SQL treat categories correctly.
# ---------------------------------------------------------
categorical_cols = [
    'Gender', 'Under_30', 'Senior_Citizen', 'Married', 'Dependents',
    'Country', 'State', 'City', 'Quarter', 'Referred_a_Friend', 'Offer',
    'Phone_Service', 'Multiple_Lines', 'Internet_Service', 'Internet_Type',
    'Online_Security', 'Online_Backup', 'Device_Protection_Plan',
    'Premium_Tech_Support', 'Streaming_TV', 'Streaming_Movies',
    'Streaming_Music', 'Unlimited_Data', 'Contract', 'Paperless_Billing',
    'Payment_Method', 'Customer_Status', 'Churn_Label', 'Churn_Category'
]
df[categorical_cols] = df[categorical_cols].astype('category')

# ---------------------------------------------------------
# 5. CREATE A BINARY CHURN FLAG (for aggregation/SQL/DAX convenience)
# Why: "Yes"/"No" strings can't be summed directly to get churn rate.
# A 1/0 flag lets us do df['Churned'].mean() = churn rate instantly.
# ---------------------------------------------------------
df['Churned'] = (df['Churn_Label'] == 'Yes').astype(int)

# ---------------------------------------------------------
# 6. OUTLIER CHECK (not removal — churn data rarely has "bad" outliers)
# Why: We check, but we do NOT remove high-value customers just because
# they're statistical outliers — they may be your most valuable segment.
# ---------------------------------------------------------
q99 = df['Monthly_Charge'].quantile(0.99)
print(f"99th percentile Monthly Charge: {q99}")
print(f"Max Monthly Charge: {df['Monthly_Charge'].max()}")
# Decision: max charge is well within a realistic telecom pricing range
# (no $10,000/month errors), so we keep all rows — this is real business
# variation, not a data error.

# ---------------------------------------------------------
# 7. FEATURE ENGINEERING
# Why: Raw tenure/age numbers are hard to use in dashboards. Bucketing
# them creates business-friendly segments stakeholders actually ask for.
# ---------------------------------------------------------

# Tenure buckets — helps answer "do new customers churn more?"
df['Tenure_Group'] = pd.cut(
    df['Tenure_in_Months'],
    bins=[0, 12, 24, 48, 60, 100],
    labels=['0-1 Year', '1-2 Years', '2-4 Years', '4-5 Years', '5+ Years']
)

# Age groups — useful for demographic segmentation in Power BI
df['Age_Group'] = pd.cut(
    df['Age'],
    bins=[18, 30, 45, 60, 100],
    labels=['19-30', '31-45', '46-60', '60+']
)

# Revenue leakage flag — customers who received refunds
# Why: Refunds are a hidden cost; flagging them lets us later measure
# how much revenue is being eroded even among retained customers.
df['Had_Refund'] = (df['Total_Refunds'] > 0).astype(int)

# ---------------------------------------------------------
# 8. FINAL SANITY CHECK
# ---------------------------------------------------------
print(df.info())
print(df.isnull().sum()[df.isnull().sum() > 0])  # should be empty now

99th percentile Monthly Charge: 114.729
Max Monthly Charge: 118.75
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 54 columns):
 #   Column                             Non-Null Count  Dtype   
---  ------                             --------------  -----   
 0   Customer_ID                        7043 non-null   object  
 1   Gender                             7043 non-null   category
 2   Age                                7043 non-null   int64   
 3   Under_30                           7043 non-null   category
 4   Senior_Citizen                     7043 non-null   category
 5   Married                            7043 non-null   category
 6   Dependents                         7043 non-null   category
 7   Number_of_Dependents               7043 non-null   int64   
 8   Country                            7043 non-null   category
 9   State                              7043 non-null   category
 10  City                               7043 n

In [ ]:
total = len(df)
churned = df['Churned'].sum()
churn_rate = 100 * churned / total
retention_rate = 100 - churn_rate

print(f"Total Customers: {total}")
print(f"Total Churned: {churned}")
print(f"Churn Rate: {churn_rate:.2f}%")
print(f"Retention Rate: {retention_rate:.2f}%")

Total Customers: 7043
Total Churned: 1869
Churn Rate: 26.54%
Retention Rate: 73.46%


In [ ]:
total_revenue = df['Total_Revenue'].sum()
lost_historical_revenue = df.loc[df['Churned']==1, 'Total_Revenue'].sum()
lost_monthly_recurring = df.loc[df['Churned']==1, 'Monthly_Charge'].sum()

print(f"Total Revenue (all customers, lifetime): ${total_revenue:,.0f}")
print(f"Revenue tied to churned customers: ${lost_historical_revenue:,.0f}")
print(f"Monthly Recurring Revenue (MRR) now lost: ${lost_monthly_recurring:,.0f}/month")

Total Revenue (all customers, lifetime): $21,371,132
Revenue tied to churned customers: $3,684,460
Monthly Recurring Revenue (MRR) now lost: $139,131/month


In [ ]:
df.groupby('Churn_Label')['Monthly_Charge'].mean()

/tmp/ipykernel_694/268647309.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby('Churn_Label')['Monthly_Charge'].mean()


,Monthly_Charge
Churn_Label,
No,61.265124
Yes,74.441332


In [ ]:
df.groupby('Gender')['Churned'].mean() * 100

/tmp/ipykernel_694/21166615.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby('Gender')['Churned'].mean() * 100


,Churned
Gender,
Female,26.920872
Male,26.160338


In [ ]:
df.groupby('Age_Group', observed=True)['Churned'].mean() * 100

,Churned
Age_Group,
19-30,22.367560
31-45,23.520329
46-60,24.305919
60+,36.462094


In [ ]:
df.groupby('Contract', observed=True)['Churned'].agg(['mean','count'])

,mean,count
Contract,,
Month-to-Month,0.458449,3610
One Year,0.107097,1550
Two Year,0.025491,1883


In [ ]:
df.groupby('Payment_Method', observed=True)['Churned'].mean().sort_values(ascending=False) * 100

,Churned
Payment_Method,
Mailed Check,36.883117
Bank Withdrawal,33.998465
Credit Card,14.477992


In [ ]:
df.groupby('Satisfaction_Score')['Churned'].mean() * 100

,Churned
Satisfaction_Score,
1,100.000000
2,100.000000
3,16.097561
4,0.000000
5,0.000000


In [ ]:
df[df['Churned']==1]['Churn_Category'].value_counts()

,count
Churn_Category,
Competitor,841
Attitude,314
Dissatisfaction,303
Price,211
Other,200
Not Churned,0


In [ ]:
df.groupby('City', observed=True)['Churned'].agg(['sum','count','mean']).sort_values('sum', ascending=False).head(10)

,sum,count,mean
City,,,
San Diego,185,285,0.649123
Los Angeles,78,293,0.266212
San Francisco,31,104,0.298077
San Jose,29,112,0.258929
Fallbrook,26,43,0.604651
Sacramento,26,108,0.240741
Temecula,22,38,0.578947
Escondido,16,51,0.313725
Long Beach,15,60,0.250000


In [ ]:
df.groupby('Tenure_Group', observed=True)['Churned'].mean() * 100

,Churned
Tenure_Group,
0-1 Year,47.438243
1-2 Years,28.710938
2-4 Years,20.388959
4-5 Years,14.423077
5+ Years,6.609808


In [ ]:
df.groupby('Referred_a_Friend')['Churned'].mean() * 100

/tmp/ipykernel_694/1347458790.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby('Referred_a_Friend')['Churned'].mean() * 100


,Churned
Referred_a_Friend,
No,32.583093
Yes,19.366853


In [ ]:
from google.colab import files

# Save to Colab
df.to_csv('telco_cleaned.csv', index=False)

# Trigger automatic download to your PC
files.download('telco_cleaned.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>